# Importing libraries

In [3]:
# Basic libraries
import pandas as pd
import numpy as np
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.multioutput import MultiOutputClassifier

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss
from memory_profiler import memory_usage


# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [4]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [5]:
ds = load_dataset("higopires/RePro-categories-multilabel")

train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()
test = ds['test'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8002 entries, 0 to 8001
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   review_text             8002 non-null   object
 1   ENTREGA                 8002 non-null   int64 
 2   OUTROS                  8002 non-null   int64 
 3   PRODUTO                 8002 non-null   int64 
 4   CONDICOESDERECEBIMENTO  8002 non-null   int64 
 5   INADEQUADA              8002 non-null   int64 
 6   ANUNCIO                 8002 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 437.7+ KB


# Dataset preprocessing

In [6]:
train = train[train['INADEQUADA'] == 0].reset_index(drop=True)
val = val[val['INADEQUADA'] == 0].reset_index(drop=True)
test = test[test['INADEQUADA'] == 0].reset_index(drop=True)

train = train.drop(columns=['INADEQUADA'])
val = val.drop(columns=['INADEQUADA'])
test = test.drop(columns=['INADEQUADA'])

In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   review_text             966 non-null    object
 1   ENTREGA                 966 non-null    int64 
 2   OUTROS                  966 non-null    int64 
 3   PRODUTO                 966 non-null    int64 
 4   CONDICOESDERECEBIMENTO  966 non-null    int64 
 5   ANUNCIO                 966 non-null    int64 
dtypes: int64(5), object(1)
memory usage: 45.4+ KB


In [8]:
train.rename(columns={'review_text': 'text'}, inplace=True)
val.rename(columns={'review_text': 'text'}, inplace=True)
test.rename(columns={'review_text': 'text'}, inplace=True)

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7674 entries, 0 to 7673
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   text                    7674 non-null   object
 1   ENTREGA                 7674 non-null   int64 
 2   OUTROS                  7674 non-null   int64 
 3   PRODUTO                 7674 non-null   int64 
 4   CONDICOESDERECEBIMENTO  7674 non-null   int64 
 5   ANUNCIO                 7674 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 359.8+ KB


In [9]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [10]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': MultiOutputClassifier(SVC()),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultiOutputClassifier(MultinomialNB()),
        'params': {
            'estimator__alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': MultiOutputClassifier(LogisticRegression(max_iter=1000)),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': MultiOutputClassifier(GradientBoostingClassifier()),
        'params': {
            'estimator__n_estimators': [100, 150, 200],
            'estimator__criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': MultiOutputClassifier(AdaBoostClassifier()),
        'params': {
            'estimator__n_estimators': [50, 100, 150],
            'estimator__learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': MultiOutputClassifier(SGDClassifier()),
        'params': {
            'estimator__alpha': [0.0001, 0.001, 0.01],
            'estimator__penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [11]:
train

,text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,ANUNCIO
0,aparelho muito bom confiável e com valor aquis...,0,0,1,0,0
1,história é muito boa porém autor enrolou um po...,0,0,1,0,0
2,entrega rápida produto muito bom amei praticidade,1,0,1,0,0
3,produto otimo falta carregador da maquina pequena,0,0,1,1,0
4,proteção anti queda não é boa se cair de frent...,0,0,1,0,0
...,...,...,...,...,...,...
7669,amei produto chegou prazo e em perfeito estado...,1,0,1,1,0
7670,ótima embalagem produto entregue prazo recomen...,1,0,1,1,0
7671,ótimo produto super recomendo entrega bem rápida,1,0,1,0,0
7672,veio tudo certinho dentro prazo e produto é mu...,1,0,1,1,0


In [12]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'hamming_loss', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = list(train.columns[1:])
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [13]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train[classes])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                val_classes = val.drop(columns=['text'])

                accuracy = accuracy_score(val_classes, y_pred)
                
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val_classes.to_numpy(), y_pred, average=None, zero_division=0)

                hamm_loss = hamming_loss(val_classes.to_numpy(), y_pred)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'hamming_loss': hamm_loss,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_multilabel1.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 5.514278099988587
Peak memory usage during training: 422.42578125 MB
Prediction time: 1.0579653999884613
Peak memory usage during prediction: 422.546875 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_16340\3004956853.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 6.051981600001454
Peak memory usage during training: 427.14453125 MB
Prediction time: 1.9433777999947779
Peak memory usage during prediction: 422.203125 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer
Training time: 6.008134900010191
Peak memory usage during training: 427.484375 MB
Prediction time: 1.4801886000204831
Peak memory usage during prediction: 422.41015625 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 100, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 10.084336799976882
Peak memory usage during training: 467.2578125 MB
Prediction time: 1.045807899965439
Peak memory usage during prediction: 467.25 MB
-----------------------------------------------------------------------

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Training time: 1.139551800035406
Peak memory usage during training: 539.6015625 MB
Prediction time: 1.294336299994029
Peak memory usage during prediction: 645.25 MB
----------------------------------------------------------------------------------------------------
Training KNeighbors with params {'n_neighbors': 3, 'algorithm': 'kd_tree'} and vectorizer TfidfVectorizer


...

Training time: 8.3894108000095
Peak memory usage during training: 478.30078125 MB
Prediction time: 1.2426844000001438
Peak memory usage during prediction: 469.4765625 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'estimator__n_estimators': 150, 'estimator__learning_rate': 0.1} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to ci

Training time: 8.37211410002783
Peak memory usage during training: 477.53125 MB
Prediction time: 1.2959291999577545
Peak memory usage during prediction: 469.671875 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'estimator__n_estimators': 150, 'estimator__learning_rate': 1.0} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to ci

Training time: 8.551284099987242
Peak memory usage during training: 478.41015625 MB
Prediction time: 1.3618023000308312
Peak memory usage during prediction: 469.76953125 MB
----------------------------------------------------------------------------------------------------
Processing model: SGD
Training SGD with params {'estimator__alpha': 0.0001, 'estimator__penalty': 'l2'} and vectorizer CountVectorizer
Training time: 1.4561576000414789
Peak memory usage during training: 477.44140625 MB
Prediction time: 1.905766899988521
Peak memory usage during prediction: 469.51953125 MB
----------------------------------------------------------------------------------------------------
Training SGD with params {'estimator__alpha': 0.0001, 'estimator__penalty': 'l1'} and vectorizer CountVectorizer
Training time: 1.0627722999779508
Peak memory usage during training: 476.7890625 MB
Prediction time: 1.91933100001188
Peak memory usage during prediction: 469.26953125 MB
---------------------------------

# Process results

In [14]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 25 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   seed                                    396 non-null    object 
 1   vectorizer                              396 non-null    object 
 2   model                                   396 non-null    object 
 3   params                                  396 non-null    object 
 4   accuracy                                396 non-null    float64
 5   hamming_loss                            396 non-null    float64
 6   training_time                           396 non-null    float64
 7   prediction_time                         396 non-null    float64
 8   peak_memory_train                       396 non-null    float64
 9   peak_memory_prediction                  396 non-null    float64
 10  precision_class_ENTREGA                 396 non-null    float6

In [15]:
results.head()

,seed,vectorizer,model,params,accuracy,hamming_loss,training_time,prediction_time,peak_memory_train,peak_memory_prediction,...,f1_class_OUTROS,precision_class_PRODUTO,recall_class_PRODUTO,f1_class_PRODUTO,precision_class_CONDICOESDERECEBIMENTO,recall_class_CONDICOESDERECEBIMENTO,f1_class_CONDICOESDERECEBIMENTO,precision_class_ANUNCIO,recall_class_ANUNCIO,f1_class_ANUNCIO
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.642857,0.106933,5.514278,1.057965,422.425781,422.546875,...,0.488136,0.872390,0.986877,0.926108,0.919355,0.360759,0.518182,1.0,0.156627,0.270833
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.626050,0.116807,6.051982,1.943378,427.144531,422.203125,...,0.443662,0.870670,0.989501,0.926290,0.914894,0.272152,0.419512,1.0,0.072289,0.134831
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.626050,0.116807,6.008135,1.480189,427.484375,422.410156,...,0.443662,0.870670,0.989501,0.926290,0.914894,0.272152,0.419512,1.0,0.072289,0.134831
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.656513,0.103782,10.084337,1.045808,467.257812,467.250000,...,0.510067,0.873843,0.990814,0.928659,0.903226,0.354430,0.509091,1.0,0.192771,0.323232
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.628151,0.113445,11.392574,1.070524,476.820312,476.808594,...,0.477509,0.872390,0.986877,0.926108,0.893617,0.265823,0.409756,1.0,0.048193,0.091954


In [16]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,hamming_loss,training_time,prediction_time,peak_memory_train,peak_memory_prediction,...,precision_class_PRODUTO,recall_class_PRODUTO,f1_class_PRODUTO,precision_class_CONDICOESDERECEBIMENTO,recall_class_CONDICOESDERECEBIMENTO,f1_class_CONDICOESDERECEBIMENTO,precision_class_ANUNCIO,recall_class_ANUNCIO,f1_class_ANUNCIO,f1_avg
0,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.577731,0.139076,5.836746,1.173982,485.425781,480.613281,...,0.834628,0.986877,0.904390,0.855072,0.373418,0.519824,1.000000,0.144578,0.252632,0.500622
1,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.637605,0.111555,5.928384,1.215968,485.651042,480.658854,...,0.839246,0.993438,0.909856,0.876404,0.493671,0.631579,1.000000,0.445783,0.616667,0.686748
2,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.692227,0.084034,6.011802,1.203333,485.584635,480.674479,...,0.906748,0.969816,0.937223,0.796610,0.594937,0.681159,0.878788,0.698795,0.778523,0.789880
3,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.579832,0.136975,8.529893,1.266062,485.924479,479.545573,...,0.834071,0.989501,0.905162,0.855072,0.373418,0.519824,1.000000,0.180723,0.306122,0.516954
4,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.654412,0.103151,8.532834,1.333554,485.686198,478.054688,...,0.843159,0.994751,0.912703,0.863158,0.518987,0.648221,1.000000,0.542169,0.703125,0.725466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'rbf'}",3.333333,0.704832,0.083824,19.001022,2.798409,643.281250,474.355469,...,0.901324,0.982940,0.940364,0.891304,0.518987,0.656000,1.000000,0.361446,0.530973,0.738595
128,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'sigm...",3.333333,0.732143,0.069748,11.078051,1.498658,558.207031,470.294271,...,0.931250,0.977690,0.953905,0.858491,0.575949,0.689394,0.979592,0.578313,0.727273,0.803399
129,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'lin...",3.333333,0.716387,0.073739,11.448185,1.503340,561.165365,470.272135,...,0.938303,0.958005,0.948052,0.782946,0.639241,0.703833,0.846154,0.662651,0.743243,0.810494
130,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'rbf'}",3.333333,0.723739,0.074160,25.610458,2.960880,665.022135,477.338542,...,0.924224,0.976378,0.949585,0.880000,0.556962,0.682171,0.975610,0.481928,0.645161,0.781463


In [17]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train[classes])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test[classes], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test[classes], y_pred, average=None, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: LogisticRegression
Best model params: {'estimator__C': 10, 'estimator__penalty': 'l2'}
Best vectorizer: CountVectorizer
Best accuracy: 0.7028985507246377

Class ENTREGA
Precision: 0.9377162629757786
Recall: 0.9093959731543624
F1: 0.9233390119250426
Support: 298

Class OUTROS
Precision: 0.8159509202453987
Recall: 0.6073059360730594
F1: 0.6963350785340314
Support: 219

Class PRODUTO
Precision: 0.9291237113402062
Recall: 0.9474375821287779
F1: 0.9381912817176318
Support: 761

Class CONDICOESDERECEBIMENTO
Precision: 0.76
Recall: 0.6951219512195121
F1: 0.7261146496815286
Support: 164

Class ANUNCIO
Precision: 0.8309859154929577
Recall: 0.7023809523809523
F1: 0.7612903225806451
Support: 84



In [18]:
with open('models/best_model_sklearn_multilabel1.pkl', 'wb') as f:
    pickle.dump(pipeline, f)